# K-Means — Segmentacion de clientes retail

---

**Autor:** Borja Mora Méndez
**Contacto:** [borja.mora.mendez@gmail.com](mailto:borja.mora.mendez@gmail.com) · [LinkedIn](https://www.linkedin.com/in/borja-mora-mendez/)
**Repositorio:** [Data Analytics Portfolio](https://github.com/BORJAMOME/Data-Analytics-Portfolio)
**Categoría:** Machine Learning · No Supervisado · Clustering · K-Means

---

**Objetivo:** Aplicar K-Means para segmentar clientes de un centro comercial segun ingresos y habito de gasto. Incluye metodo del codo, silhouette score y visualizacion de centroides.

**Contexto de negocio:** Un centro comercial quiere agrupar a sus clientes para disenar campanas de marketing diferenciadas. La hipotesis: hay perfiles de gasto claramente distintos que se pueden explotar comercialmente.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.colors import LinearSegmentedColormap
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score

sns.set_style("whitegrid")
np.random.seed(42)

# Estilo visual — sistema de color validado (consejo UX/UI Data)
BACKGROUND    = '#fbfbfb'
PURPLE        = '#7a7bff'   # único color de énfasis / serie única en scatter y líneas (1 por gráfico)
PURPLE_LIGHT  = '#9b9cff'   # EDA de una sola serie (histogramas independientes)
POSITIVE      = '#6b8158'   # exclusivo signo positivo
NEGATIVE      = '#c34031'   # exclusivo signo negativo
NEUTRAL_BAR   = '#d9d9d9'   # barras/áreas de contexto (siempre con etiqueta de valor)
NEUTRAL_LINE  = '#8f8c9e'   # líneas de contexto
CONTEXT_LINES = [NEUTRAL_LINE, '#a89a8a', '#7d94a8']   # gama fija para 2+ líneas de contexto
INK           = '#111111'
MUTED         = '#707070'

# Paleta categórica para identidad de cluster — validada (ΔE OKLab, simulación CVD) para
# pares adyacentes (barras, líneas, enlaces de dendrograma). En scatter/PCA con 4+ clusters
# el color por sí solo no basta para daltonismo severo: por eso cada cluster lleva también
# una forma de marcador distinta (CLUSTER_MARKERS) — nunca dependas solo del color.
CLUSTER_PALETTE = ['#7a7bff', '#eb6834', '#1baf7a', '#e34948', '#eda100', '#e87ba4', '#008300']
CLUSTER_MARKERS = ['o', 's', '^', 'D', 'v', 'P', 'X']

DIVERGING_CMAP = LinearSegmentedColormap.from_list(
    "borja_diverging", ["#c34031", "#e0a89f", "#f0ede8", "#b7c2a9", "#6b8158"]
)
SEQUENTIAL_GREEN = LinearSegmentedColormap.from_list(
    "borja_sequential", [BACKGROUND, POSITIVE]
)

def color_annotations(ax, values, threshold, dark="#ffffff", light=INK):
    """Recolorea el texto de un heatmap celda a celda según su magnitud."""
    for text, value in zip(ax.texts, np.asarray(values).flatten()):
        text.set_color(dark if abs(value) >= threshold else light)

plt.rcParams.update({
    'figure.figsize': (10, 5),
    'figure.dpi': 100,
    'figure.facecolor': BACKGROUND,
    'axes.facecolor': BACKGROUND,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.edgecolor': MUTED,
    'axes.labelcolor': INK,
    'axes.titlesize': 13,
    'axes.titleweight': 'bold',
    'axes.titlecolor': INK,
    'xtick.color': MUTED,
    'ytick.color': MUTED,
    'font.family': 'sans-serif',
    'font.size': 10,
    'grid.color': '#f0f0f0',
    'grid.linewidth': 0.5,
})

print("Librerias cargadas correctamente")

## 1. Dataset sintetico: clientes de centro comercial

Generamos un dataset inspirado en el clasico Mall Customers con 200 clientes y 5 segmentos naturales.

In [ ]:
# Simular 5 segmentos naturales de clientes
segmentos = {
    "Bajo ingreso, bajo gasto":  (25, 20, 40, 8),
    "Bajo ingreso, alto gasto":  (25, 75, 40, 10),
    "Ingreso medio, gasto medio": (55, 50, 40, 12),
    "Alto ingreso, bajo gasto":  (85, 20, 40, 8),
    "Alto ingreso, alto gasto":  (85, 80, 40, 10),
}

datos = []
for nombre, (ing_mean, gasto_mean, n, std) in segmentos.items():
    ing = np.random.normal(ing_mean, std, n)
    gasto = np.random.normal(gasto_mean, std, n)
    datos.append(np.column_stack([ing, gasto]))

X_raw = np.vstack(datos)
df = pd.DataFrame(X_raw, columns=["Ingreso_Anual_K", "Spending_Score"])
df = df.clip(lower=1, upper=100).round(0).astype(int)

print(f"Dataset: {len(df)} clientes, 2 variables")
display(df.describe())

Dataset: 200 clientes, 2 variables


,Ingreso_Anual_K,Spending_Score
count,200.000000,200.000000
mean,54.505000,50.125000
std,28.213293,27.739082
min,6.000000,1.000000
25%,26.750000,23.000000
50%,56.500000,53.500000
75%,81.000000,76.000000
max,100.000000,100.000000


## 2. Exploracion visual

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

axes[0].scatter(df["Ingreso_Anual_K"], df["Spending_Score"], color=PURPLE, alpha=0.5, s=30, edgecolors="k", linewidths=0.3)
axes[0].set_xlabel("Ingreso anual (k)")
axes[0].set_ylabel("Spending Score")
axes[0].set_title("Clientes sin segmentar")

axes[1].hist(df["Ingreso_Anual_K"], bins=20, color=PURPLE_LIGHT, edgecolor='white')
axes[1].set_xlabel("Ingreso anual (k)")
axes[1].set_title("Distribucion de ingresos")

axes[2].hist(df["Spending_Score"], bins=20, color=PURPLE_LIGHT, edgecolor='white')
axes[2].set_xlabel("Spending Score")
axes[2].set_title("Distribucion de gasto")

plt.tight_layout()
plt.show()

## 3. Estandarizacion

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df)
print("Datos estandarizados (media=0, std=1)")

Datos estandarizados (media=0, std=1)


## 4. Metodo del codo + Silhouette Score

Dos criterios complementarios para elegir k:
- **Inercia (codo):** suma de distancias al cuadrado de cada punto a su centroide. Buscar donde deja de bajar significativamente.
- **Silhouette:** mide cohesion interna vs separacion entre clusters. Mas alto = mejor.

In [ ]:
K_range = range(2, 11)
inertias = []
silhouettes = []

for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_scaled)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X_scaled, km.labels_))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(K_range, inertias, marker="o", linewidth=2, color=PURPLE)
axes[0].set_xlabel("k")
axes[0].set_ylabel("Inercia")
axes[0].set_title("Metodo del codo")
axes[0].axvline(x=5, color=INK, linestyle="--", alpha=0.6, label="k=5")
axes[0].legend()

axes[1].plot(K_range, silhouettes, marker="s", linewidth=2, color=PURPLE)
axes[1].set_xlabel("k")
axes[1].set_ylabel("Silhouette Score")
axes[1].set_title("Silhouette Score")
best_k = K_range[np.argmax(silhouettes)]
axes[1].axvline(x=best_k, color=INK, linestyle="--", alpha=0.6, label=f"Mejor: k={best_k}")
axes[1].legend()

plt.tight_layout()
plt.show()

print(f"Mejor silhouette: k={best_k} ({max(silhouettes):.4f})")

## 5. Modelo final y visualizacion de centroides

In [ ]:
kmeans = KMeans(n_clusters=5, random_state=42, n_init=10)
df["Cluster"] = kmeans.fit_predict(X_scaled)

# Centroides en espacio original
centroids_orig = scaler.inverse_transform(kmeans.cluster_centers_)

fig, ax = plt.subplots(figsize=(10, 7))

for cl in range(5):
    sub = df[df["Cluster"] == cl]
    ax.scatter(sub["Ingreso_Anual_K"], sub["Spending_Score"],
               s=50, alpha=0.6, color=CLUSTER_PALETTE[cl], marker=CLUSTER_MARKERS[cl],
               label=f"Cluster {cl} (n={len(sub)})",
               edgecolors="k", linewidths=0.3)

# Centroides
ax.scatter(centroids_orig[:, 0], centroids_orig[:, 1],
           s=200, c=INK, marker="X", zorder=5, label="Centroides")

ax.set_xlabel("Ingreso anual (k)", fontsize=12)
ax.set_ylabel("Spending Score", fontsize=12)
ax.set_title("Segmentacion K-Means: 5 clusters con centroides", fontsize=14, fontweight="bold")
ax.legend(loc="upper left")
plt.tight_layout()
plt.show()

## 6. Perfil de cada cluster

In [ ]:
perfil = df.groupby("Cluster").agg(
    n_clientes=("Ingreso_Anual_K", "count"),
    ingreso_medio=("Ingreso_Anual_K", "mean"),
    gasto_medio=("Spending_Score", "mean")
).round(1)

# Etiquetar segmentos
nombres = {0: "Prudente alto ingreso", 1: "Aspiracional", 2: "Medio equilibrado",
            3: "Prudente bajo ingreso", 4: "Premium"}

perfil["Etiqueta"] = perfil.index.map(lambda x: nombres.get(x, ""))

print("Perfil de segmentos:")
display(perfil)

print("\nInterpretacion de negocio:")
for cl in sorted(df["Cluster"].unique()):
    sub = df[df["Cluster"] == cl]
    print(f"  Cluster {cl}: ingreso={sub['Ingreso_Anual_K'].mean():.0f}k, "
          f"gasto={sub['Spending_Score'].mean():.0f}")

Perfil de segmentos:


,n_clientes,ingreso_medio,gasto_medio,Etiqueta
Cluster,,,,
0,41,83.6,82.0,Prudente alto ingreso
1,42,83.8,22.7,Aspiracional
2,42,24.3,20.2,Medio equilibrado
3,39,24.6,75.5,Prudente bajo ingreso
4,36,54.8,53.3,Premium



Interpretacion de negocio:
  Cluster 0: ingreso=84k, gasto=82
  Cluster 1: ingreso=84k, gasto=23
  Cluster 2: ingreso=24k, gasto=20
  Cluster 3: ingreso=25k, gasto=75
  Cluster 4: ingreso=55k, gasto=53


## 7. Conclusiones y recomendaciones

**Hallazgos:**
- K-Means identifica 5 segmentos de clientes con perfiles claros de ingreso-gasto.
- Los centroides resumen cada cluster en un unico punto representativo — util para targeting automatizado.
- El silhouette score confirma que la separacion entre clusters es buena.

**Recomendaciones para el centro comercial:**
- **Alto ingreso + alto gasto:** Programa VIP, exclusividad, experiencias premium.
- **Alto ingreso + bajo gasto:** Oportunidad no explotada — campanas para activar gasto.
- **Bajo ingreso + alto gasto:** Perfil aspiracional — financiacion, descuentos por volumen.
- **Bajo ingreso + bajo gasto:** Ofertas de entrada, descuentos agresivos.
- **Ingreso medio:** Programas de puntos, fidelizacion.

